In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Cài đặt các thư viện bắt buộc cho Pipeline
!pip install -q faiss-cpu
!pip install -q open_clip_torch
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q bert_score
!pip install -q datasets
!pip install -q accelerate
!pip install -q scikit-learn

print("✅ Đã cài đặt xong tất cả các thư viện cần thiết!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00
✅ Đã cài đặt xong tất cả các thư viện cần thiết!


In [3]:
import torch
import time
import evaluate
import numpy as np
import gc
import random
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
from PIL import Image, ImageFile
from transformers import pipeline

# Ép hệ thống bỏ qua lỗi ảnh thiếu byte của PathVQA
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Cố định đúng Seed 42 để đảm bảo quét trúng 500 mẫu y hệt như lượt chạy RAG
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# --- 1. Lớp VLM Generator (Bản 7B - KHÔNG DÙNG RAG / ZERO-SHOT) ---
class BaselineVlmGenerator:
    def __init__(self):
        print("\n[Hạ tầng] Đang tải Qwen2-VL-7B-Instruct (Baseline No RAG)...")
        # Sử dụng device_map="auto" để chia tải tối ưu trên Multi-GPU
        self.pipe = pipeline(
            "image-text-to-text",
            model="Qwen/Qwen2-VL-7B-Instruct",
            device_map="auto",
            model_kwargs={"torch_dtype": torch.bfloat16} 
        )
        print("✅ Đã nạp thành công mô hình cấu hình lớn 7B!")

    def optimizeImage(self, pilImg, maxRes=336):
        if pilImg.mode != "RGB": 
            pilImg = pilImg.convert("RGB")
        pilImg.thumbnail((maxRes, maxRes), Image.Resampling.LANCZOS)
        return pilImg

    def generate(self, testImg, testQuest):
        contentBlock = []
        
        # 💡 Prompt được tinh giản: Yêu cầu mô hình trả lời trực tiếp không qua tham khảo
        systemInstruction = (
            "<System>\n"
            "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
            "based solely on the provided pathology image.\n"
            "Strictly follow these rules:\n"
            "1. Analyze the provided [Image] and [Question].\n"
            "2. Provide a concise and definitive answer.\n"
            "</System>\n\n"
            "<User>\n"
        )
        contentBlock.append({"type": "text", "text": systemInstruction})
        
        # Chỉ nạp ảnh Test
        optimizedTestImg = self.optimizeImage(testImg)
        contentBlock.append({"type": "image", "image": optimizedTestImg})
        
        targetInstruction = (
            f"Question: {testQuest}\n\n"
            "Based on the visual evidence, what is the correct answer?\n"
            "</User>\n\n"
            "<Assistant>\n"
            "Final Answer: "
        )
        contentBlock.append({"type": "text", "text": targetInstruction})
        
        messages = [{"role": "user", "content": contentBlock}]
        outputs = self.pipe(
            messages, 
            max_new_tokens=15, 
            generate_kwargs={"do_sample": False, "max_length": None}
        )
        
        rawResponse = outputs[0]['generated_text']
        finalText = rawResponse[-1]['content'] if isinstance(rawResponse, list) else rawResponse
        return finalText.replace("</Assistant>", "").strip()

# --- 2. Thực thi Pipeline Đánh giá Baseline ---
def runKagglePathVqa7bZeroShotBenchmark(sampleSize=500):
    print("1. Khởi tạo các thư viện đo lường (Metrics)...")
    bleuMetric = evaluate.load("bleu")
    rougeMetric = evaluate.load("rouge")
    bertscoreMetric = evaluate.load("bertscore")
    
    print("2. Tải mô hình và kho dữ liệu PathVQA (Chỉ lấy tập Test)...")
    vlm = BaselineVlmGenerator()
    testDS = load_dataset("flaviagiammarino/path-vqa", split="test")
    
    # 💡 Lấy ngẫu nhiên mẫu trùng khớp hoàn toàn với tập mẫu 500 của bản RAG
    totalIndices = list(range(len(testDS)))
    sampledIndices = random.sample(totalIndices, min(sampleSize, len(testDS)))
    
    results = {"closed": {"ref": [], "pred": []}, "open": {"ref": [], "pred": []}}
    latencyList = []
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    print(f"\n🚀 BẮT ĐẦU CHẠY THỰC NGHIỆM ZERO-SHOT BASELINE (QWEN-7B KHÔNG RAG) TRÊN {len(sampledIndices)} MẪU...")
    print("-" * 60)
    
    if torch.cuda.is_available(): 
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
    startTotalTime = time.time()

    for progressCount, idx in enumerate(sampledIndices):
        item = testDS[idx]
        startTimeSample = time.time()
        
        try:
            # 💡 Chỉ gọi vlm.generate với ảnh và câu hỏi, không truyền contexts/trainDS
            aiAnswerRaw = vlm.generate(item['image'], item['question'])
        except Exception:
            continue
            
        latencyList.append(time.time() - startTimeSample)
        
        aiAnswer = aiAnswerRaw.lower().strip()
        groundTruth = str(item['answer']).lower().strip()
        
        if groundTruth in ['yes', 'no']:
            results["closed"]["ref"].append(groundTruth)
            hasYes = any(word in aiAnswer for word in yesKeywords)
            hasNo = any(word in aiAnswer for word in noKeywords)
            
            if hasYes and not hasNo:
                predClean = 'yes'
            elif hasNo and not hasYes:
                predClean = 'no'
            else:
                firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
            results["closed"]["pred"].append(predClean)
        else:
            results["open"]["ref"].append(groundTruth)
            results["open"]["pred"].append(aiAnswer)
            
        if (progressCount + 1) % 50 == 0: 
            print(f"Tiến độ Kaggle Baseline (7B): {progressCount + 1}/{len(sampledIndices)} mẫu...")

    endTotalTime = time.time()
    
    # --- XUẤT BÁO CÁO THỰC NGHIỆM CHÍNH THỨC ---
    print("\n" + "="*60 + "\n📊 BÁO CÁO THỰC NGHIỆM - ZERO-SHOT BASELINE (QWEN-7B NO RAG ON PATHVQA)\n" + "="*60)
    
    if results["closed"]["ref"]:
        accScore = accuracy_score(results["closed"]["ref"], results["closed"]["pred"])
        f1Weighted = f1_score(results["closed"]["ref"], results["closed"]["pred"], average='weighted', zero_division=0)
        print(f"[1. Closed-Ended Classification (Total: {len(results['closed']['ref'])} samples)]")
        print(f"  - Accuracy : {accScore:.4f}")
        print(f"  - F1-Score : {f1Weighted:.4f}")
    
    if results["open"]["ref"]:
        print(f"\n[2. Open-Ended Text Generation (Total: {len(results['open']['ref'])} samples)]")
        predsList = results["open"]["pred"]
        refsList = [[r] for r in results["open"]["ref"]]
        
        bleu1 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=1, smooth=True)['bleu']
        bleu2 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=2, smooth=True)['bleu']
        bleu3 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=3, smooth=True)['bleu']
        rougeL = rougeMetric.compute(predictions=predsList, references=results["open"]["ref"])['rougeL']
        
        bertResults = bertscoreMetric.compute(predictions=predsList, references=results["open"]["ref"], lang="en")
        print(f"  - BLEU-1   : {bleu1:.4f}")
        print(f"  - BLEU-2   : {bleu2:.4f}")
        print(f"  - BLEU-3   : {bleu3:.4f}")
        print(f"  - ROUGE-L  : {rougeL:.4f}")
        print(f"  - BERTScore: {np.mean(bertResults['f1']):.4f}")

    totalDuration = endTotalTime - startTotalTime
    print("\n[3. Computational Efficiency Measurements]")
    print(f"  - Inference Latency (Độ trễ) : {np.mean(latencyList):.2f} s/mẫu")
    print(f"  - Throughput (Tốc độ)        : {len(latencyList) / totalDuration:.2f} mẫu/giây")
    
    if torch.cuda.is_available():
        print("\n[4. Multi-GPU Memory Profile]")
        for gpuId in range(torch.cuda.device_count()):
            vramMB = torch.cuda.max_memory_allocated(device=gpuId) / (1024 ** 2)
            print(f"  - Peak Memory Usage on GPU {gpuId}     : {vramMB:.2f} MB")
    print("="*60)
    
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if __name__ == "__main__":
    runKagglePathVqa7bZeroShotBenchmark(sampleSize=500)

1. Khởi tạo các thư viện đo lường (Metrics)...


2. Tải mô hình và kho dữ liệu PathVQA (Chỉ lấy tập Test)...

[Hạ tầng] Đang tải Qwen2-VL-7B-Instruct (Baseline No RAG)...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

... (36 outputs left out)

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  - BLEU-1   : 0.0217
  - BLEU-2   : 0.0088
  - BLEU-3   : 0.0042
  - ROUGE-L  : 0.0474
  - BERTScore: 0.8383

[3. Computational Efficiency Measurements]
  - Inference Latency (Độ trễ) : 1.99 s/mẫu
  - Throughput (Tốc độ)        : 0.50 mẫu/giây

[4. Multi-GPU Memory Profile]
  - Peak Memory Usage on GPU 0     : 8307.83 MB
  - Peak Memory Usage on GPU 1     : 8652.34 MB
